# Spore Agentic Decompiler: external-memory ablation study

**36 controlled trials, 3 known-solvable functions × 4 retrieval arms × 3 repetitions**

This notebook recomputes metrics from the **actual per-trial records**, including failed runs, instead of relying on the successful-runs-only summary. It displays reproducible figures and produces a shareable ZIP containing an HTML dashboard, data tables, figures and a redacted evidence export.

**Four treatments:** baseline (no retrieval), raw (similar historical findings), consolidated (findings and compiler techniques), and placebo (deterministically chosen *real* findings with low lexical overlap). Placebo does not mean fabricated text. The evidence may still be broadly relevant. No `codegen.md` comparison is isolated by this design.

**Scientific limits:** Only three functions, nine trials per arm. Success is the original byte-matching endpoint as recorded by the suite, not a guarantee of complete semantic equivalence. This is exploratory analysis; descriptive differences are real for these runs but their causes are not established. Failure costs must count toward efficiency metrics.

**Requirements:** Python 3.10+, `jupyterlab` or `notebook`, `pandas`, `numpy`, and `matplotlib`. Install once if needed with `python3 -m pip install jupyterlab pandas numpy matplotlib`.

Open from the repository checkout. The notebook runs local, read-only log extraction, refreshes the suite's generated summary files (no models or API calls), and writes its own results in `~/sad-brain-study/`. You can override paths with `SAD_SUITE_ROOT`, `SAD_STUDY_OUTPUT`, and, to analyse an existing exported JSON without a suite checkout, `SAD_ABLATION_EXPORT`.

In [ ]:
import os, sys, json, subprocess, shutil, math, html, zipfile
from pathlib import Path
from collections import Counter

try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
    from IPython.display import display, Markdown, HTML
except ImportError as exc:
    raise RuntimeError("Install notebook dependencies: python3 -m pip install jupyterlab pandas numpy matplotlib") from exc

ARMS = ["baseline", "raw", "consolidated", "placebo"]
LABELS = {"baseline":"Baseline", "raw":"Raw memory",
          "consolidated":"Consolidated", "placebo":"Placebo"}
COLORS = {"baseline":"#56616c", "raw":"#2176b5", "consolidated":"#8157ad", "placebo":"#da7b30"}
plt.rcParams.update({"figure.figsize":(11, 5.4), "figure.dpi":110,
                     "savefig.dpi":180, "axes.spines.top":False, "axes.spines.right":False,
                     "axes.titlesize":14, "font.size":10, "grid.alpha":0.23})

candidates = [Path.cwd(), Path.cwd().parent]
REPO = next((p.resolve() for p in candidates if (p/"sad_suite_export.py").is_file()), None)
EXPORT_ENV = os.environ.get("SAD_ABLATION_EXPORT")
SUITE = Path(os.environ.get("SAD_SUITE_ROOT", str(Path.home()/"sad-known-solved-suite-fast"))).expanduser()
OUT = Path(os.environ.get("SAD_STUDY_OUTPUT", str(Path.home()/"sad-brain-study"))).expanduser().resolve()
FIGURES = OUT/"figures"
OUT.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
PLOTS = []

def saveplot(fig, slug, description):
    path = FIGURES/(slug+".png")
    fig.savefig(path, dpi=180, bbox_inches="tight", facecolor="white")
    PLOTS.append({"file":path.name,"description":description})
    display(fig)
    plt.close(fig)

def fmtint(x):
    return f"{x:,.0f}" if pd.notna(x) and np.isfinite(float(x)) else "n/a"

def pct(x):
    return f"{x:.1%}" if pd.notna(x) and np.isfinite(float(x)) else "n/a"

print("Repository:", REPO or "not required for existing export")
print("Suite:", SUITE)
print("Outputs:", OUT)

## 1. Collect and validate evidence

The notebook calls the repository's existing `sad_suite_export.py` to parse per-trial OpenCode JSONL, trial plans, result ledgers and retrieval traces. It does **not** run agents, call models or alter frozen experiments. The suite's `compare` step only refreshes derived report and CSV files. A pre-exported evidence JSON can be supplied instead via the environment variable above.

In [ ]:
EVIDENCE = OUT/"evidence.json"
if EXPORT_ENV:
    source = Path(EXPORT_ENV).expanduser().resolve()
    if not source.is_file():
        raise FileNotFoundError(source)
    if source != EVIDENCE:
        shutil.copy2(source, EVIDENCE)
    print("Loaded existing export:", source)
else:
    if REPO is None:
        raise FileNotFoundError("Launch from the SAD checkout or set SAD_ABLATION_EXPORT")
    if not (SUITE/"suite-plan.json").is_file():
        raise FileNotFoundError(f"Suite not found at {SUITE}; set SAD_SUITE_ROOT")
    for cmd in (
        [sys.executable, str(REPO/"brain_ablation_suite.py"), "compare", "--out", str(SUITE)],
        [sys.executable, str(REPO/"sad_suite_export.py"), "--suite", str(SUITE), "--output", str(EVIDENCE)]
    ):
        p = subprocess.run(cmd, cwd=str(REPO), capture_output=True, text=True)
        if p.returncode:
            raise RuntimeError(f"Command failed: {' '.join(cmd)}\n{p.stdout[-3000:]}\n{p.stderr[-3000:]}")
        print(p.stdout.strip().splitlines()[-1])

evidence = json.loads(EVIDENCE.read_text(encoding="utf-8"))
raw = evidence["trials"]
expected = len(evidence["suite"]["targets"]) * len(ARMS) * int(evidence["suite"]["repeats"])
if len(raw) != expected:
    raise ValueError(f"Incomplete report: {len(raw)}/{expected} trials")
assert len({r["name"] for r in raw}) == len(raw), "Duplicate trial identifier"
print(f"Evidence loaded: {len(raw)}/{expected} trials, {len(evidence.get('warnings', []))} exporter warnings")

In [ ]:
rows = []
steps = []
retrievals = []
for r in raw:
    usage = (r.get("logs") or {}).get("usage") or {}
    valid = r.get("valid") is True and r.get("state")=="completed" and r.get("match") is not None
    row = {k:r.get(k) for k in ("name", "target", "arm", "repeat", "state", "status",
          "match", "valid", "checker_calls", "brain_queries", "findings_returned",
          "techniques_returned", "treatment_adherent", "exposure", "diff_bytes", "seed",
          "schedule_position", "elapsed_seconds", "started_utc", "finished_utc", "exit_code", "result_note")}
    row.update(valid_trial=valid, success=int(r.get("match") is True) if valid else np.nan,
               input_tokens=usage.get("input", r.get("input_tokens")),
               output_tokens=usage.get("output", r.get("output_tokens")),
               cache_read_tokens=usage.get("cache_read"),
               cache_write_tokens=usage.get("cache_write"),
               cost_db_usd=r.get("cost_usd"), cost_steps_usd=usage.get("opencode_cost_usd"),
               model_steps=(r.get("logs") or {}).get("model_steps"),
               tool_uses=sum(((r.get("logs") or {}).get("tool_counts") or {}).values()),
               unparseable_lines=(r.get("logs") or {}).get("unparseable_lines"))
    rows.append(row)
    for s in (r.get("logs") or {}).get("steps", []):
        steps.append({"name":r["name"],"target":r["target"],"arm":r["arm"],
                      "success":row["success"], **s})
    for j,q in enumerate(r.get("retrieval_queries") or [], 1):
        retrievals.append({"name":r["name"],"target":r["target"],"arm":r["arm"],
                           "success":row["success"],"query_number":j,**q})

df = pd.DataFrame(rows)
for col in ("input_tokens","output_tokens","cache_read_tokens","cache_write_tokens",
            "cost_db_usd","cost_steps_usd","checker_calls","model_steps","tool_uses",
            "elapsed_seconds","brain_queries","findings_returned","techniques_returned",
            "schedule_position","repeat"):
    df[col] = pd.to_numeric(df[col], errors="coerce")
df["tokens"] = df["input_tokens"] + df["output_tokens"]
df["all_token_units"] = df["tokens"] + df["cache_read_tokens"].fillna(0) + df["cache_write_tokens"].fillna(0)
df["outcome"] = np.where(~df["valid_trial"], "INVALID", np.where(df["success"]==1,"MATCH","NO MATCH"))
df["arm"] = pd.Categorical(df["arm"],categories=ARMS,ordered=True)
valid = df[df["valid_trial"]].copy()
step_df = pd.DataFrame(steps)
query_df = pd.DataFrame(retrievals)

counts = df.groupby(["target","arm"],observed=True).agg(n=("name","count"),valid=("valid_trial","sum"),successes=("success","sum"))
display(counts)
assert set(df["arm"].astype(str)) == set(ARMS), "One or more arms missing"
if not counts["n"].eq(int(evidence["suite"]["repeats"])).all():
    print("WARNING: Unequal target/arm replication")
if len(valid)<len(df):
    print(f"WARNING: {len(df)-len(valid)} invalid/incomplete trials excluded from primary metrics")
if evidence.get("warnings"):
    display(pd.DataFrame({"export_warning":evidence["warnings"]}).head(25))
print("Missing token values:", int(valid["tokens"].isna().sum()))
print("Missing timing values:", int(valid["elapsed_seconds"].isna().sum()))

## 2. Primary endpoint and whole-experiment efficiency

**Success** is the verified MATCH recorded by the suite. **Tokens per successful run** conditions on succeeding and excludes the cost of failures. Our primary efficiency comparison instead uses **all input + output tokens spent across valid attempts divided by the number of MATCHes**. This is a descriptive cost-per-success metric. Cached reads/writes are reported *separately*, as tokens billed in different categories are not interchangeable at face value.

The confidence intervals in the success chart are Wilson 95% binomial intervals, describing uncertainty for these small samples, not proof of generalization to unseen functions.

In [ ]:
def wilson(k, n, z=1.959963984540054):
    if n<=0: return (np.nan,np.nan)
    p=k/n; den=1+z*z/n
    centre=(p+z*z/(2*n))/den
    delta=z*math.sqrt((p*(1-p)+z*z/(4*n))/n)/den
    return (max(0,centre-delta),min(1,centre+delta))

agg = (valid.groupby("arm", observed=True).agg(
    trials=("success","count"),matches=("success","sum"),
    total_input=("input_tokens","sum"),total_output=("output_tokens","sum"),
    total_cache_read=("cache_read_tokens","sum"),total_cache_write=("cache_write_tokens","sum"),
    total_tokens=("tokens","sum"),mean_tokens_all=("tokens","mean"),
    median_tokens_all=("tokens","median"),total_checker_calls=("checker_calls","sum"),
    mean_checker_calls=("checker_calls","mean"),total_minutes=("elapsed_seconds",lambda x: x.sum()/60),
    mean_minutes=("elapsed_seconds",lambda x: x.mean()/60),
    db_cost_total=("cost_db_usd","sum"),model_cost_total=("cost_steps_usd","sum"),
    retrieval_queries=("brain_queries","sum"),model_steps=("model_steps","sum")
).reindex(ARMS))
agg["success_rate"] = agg["matches"]/agg["trials"]
agg["tokens_per_match_all_attempts"] = agg["total_tokens"]/agg["matches"].replace(0,np.nan)
agg["tokens_per_successful_run"] = valid[valid["success"]==1].groupby("arm",observed=True)["tokens"].mean().reindex(ARMS)
agg["checker_calls_per_match_all_attempts"] = agg["total_checker_calls"]/agg["matches"].replace(0,np.nan)
agg["cost_steps_per_match"] = agg["model_cost_total"]/agg["matches"].replace(0,np.nan)
agg["cost_db_per_match"] = agg["db_cost_total"]/agg["matches"].replace(0,np.nan)
agg["wilson_low"] = [wilson(int(row.matches),int(row.trials))[0] for row in agg.itertuples()]
agg["wilson_high"] = [wilson(int(row.matches),int(row.trials))[1] for row in agg.itertuples()]
agg["delta_success_pp_vs_baseline"] = (agg["success_rate"]-agg.loc["baseline","success_rate"])*100
agg["token_efficiency_delta_vs_baseline_pct"] = (agg["tokens_per_match_all_attempts"] / agg.loc["baseline","tokens_per_match_all_attempts"]-1)*100

display(agg[["matches","trials","success_rate","total_tokens","mean_tokens_all",
             "tokens_per_successful_run","tokens_per_match_all_attempts",
             "total_cache_read","total_checker_calls","mean_minutes"]].round(2))

# A report-ready overview of the main comparisons
for arm in ARMS[1:]:
    v = agg.loc[arm]
    print(f"{LABELS[arm]} vs baseline: {v['delta_success_pp_vs_baseline']:+.1f} percentage points success, "
          f"{v['token_efficiency_delta_vs_baseline_pct']:+.1f}% all-attempt tokens per MATCH")

In [ ]:
fig,ax=plt.subplots(figsize=(10,5))
xs=np.arange(len(ARMS)); s=agg["success_rate"].to_numpy()
ylo=agg["wilson_low"].to_numpy(); yhi=agg["wilson_high"].to_numpy()
ax.bar(xs, 100*s, color=[COLORS[a] for a in ARMS],width=.6)
ax.errorbar(xs,100*s,yerr=[100*(s-ylo),100*(yhi-s)],fmt="none",color="#222",capsize=5,lw=1.6)
for i,a in enumerate(ARMS):
    ax.text(i,100*s[i]+2.4,f"{int(agg.loc[a,'matches'])}/{int(agg.loc[a,'trials'])}",ha="center",fontweight="bold")
ax.set_xticks(xs,[LABELS[a] for a in ARMS]);ax.set_ylim(0,112)
ax.set_ylabel("Verified MATCH rate (%)");ax.set_title("Primary endpoint: success by treatment")
ax.grid(axis="y")
saveplot(fig,"01-success-rates","MATCH rate and Wilson 95% intervals; n=9 per arm")

In [ ]:
fig,ax=plt.subplots(figsize=(11,5.5))
xs=np.arange(len(ARMS));w=.35
z1=agg["tokens_per_successful_run"].to_numpy()/1000
z2=agg["tokens_per_match_all_attempts"].to_numpy()/1000
ax.bar(xs-w/2,z1,w,label="Successful runs only",color="#b3bdc8")
ax.bar(xs+w/2,z2,w,label="All attempts / MATCH",color=[COLORS[a] for a in ARMS])
for i in xs:
    for xx,yy in [(i-w/2,z1[i]),(i+w/2,z2[i])]:
        if np.isfinite(yy): ax.text(xx,yy+2,f"{yy:.0f}k",ha="center",fontsize=9)
ax.set_xticks(xs,[LABELS[a] for a in ARMS]);ax.set_ylabel("Input + output tokens (thousands)")
ax.set_title("The efficiency denominator matters: include failed attempts")
ax.legend(frameon=False);ax.grid(axis="y");ax.set_ylim(0,np.nanmax(np.r_[z1,z2])*1.2)
saveplot(fig,"02-total-efficiency","Successful-run average versus total spend divided by all successful matches")

## 3. Function-specific effects and trial variability

If a condition helps one function but hurts another, pooled performance can obscure the interaction. The plots below keep all individual outcomes visible. Each small dot is one independent trial. The token plot explicitly distinguishes success from failure and does not discard failed-run tokens.

In [ ]:
func = valid.groupby(["target","arm"],observed=True).agg(
    trials=("success","count"),wins=("success","sum"),mean_tokens=("tokens","mean"),
    sum_tokens=("tokens","sum"),mean_checks=("checker_calls","mean"),
    mean_steps=("model_steps","mean")).reset_index()
func["win_rate"]=func["wins"]/func["trials"]
func["tokens_per_match"] = func["sum_tokens"]/func["wins"].replace(0,np.nan)
display(func.round(2))

matrix=func.pivot(index="target",columns="arm",values="win_rate").reindex(columns=ARMS)
fig,ax=plt.subplots(figsize=(10,4.4))
img=ax.imshow(matrix.to_numpy(),cmap="Blues",vmin=0,vmax=1,aspect="auto")
for i,t in enumerate(matrix.index):
    for j,a in enumerate(ARMS):
        rr=func[(func["target"]==t)&(func["arm"]==a)].iloc[0]
        ax.text(j,i,f"{int(rr.wins)}/{int(rr.trials)}",ha="center",va="center",
                color="white" if rr.win_rate>.68 else "#263744",fontweight="bold",fontsize=12)
ax.set_xticks(np.arange(len(ARMS)),[LABELS[a] for a in ARMS]);ax.set_yticks(np.arange(len(matrix.index)),matrix.index)
ax.set_title("Function × treatment: number of verified matches")
fig.colorbar(img,ax=ax,label="Success fraction",fraction=.035,pad=.03)
saveplot(fig,"03-function-success-heatmap","Function-stratified success counts; 3 repetitions each")

In [ ]:
fig,ax=plt.subplots(figsize=(11,5.5))
rng=np.random.default_rng(107)
for i,a in enumerate(ARMS):
    sub=valid[valid["arm"]==a]
    yy=sub["tokens"].to_numpy()/1000
    offsets=rng.uniform(-.16,.16,len(sub))
    ok=sub["success"].to_numpy()==1
    ax.scatter(i+offsets[ok],yy[ok],s=76,c=COLORS[a],marker="o",edgecolors="white",linewidths=.75,zorder=4)
    ax.scatter(i+offsets[~ok],yy[~ok],s=95,c=COLORS[a],marker="x",linewidths=2,zorder=4)
    ax.hlines(np.mean(yy),i-.22,i+.22,colors="#212b36",linewidth=2.5,zorder=3)
ax.set_xticks(np.arange(len(ARMS)),[LABELS[a] for a in ARMS]);ax.set_ylabel("Input + output tokens (thousands)")
ax.set_title("All 36 trials: cost variation and failures")
ax.grid(axis="y")
ax.legend(handles=[Line2D([0],[0],marker="o",color="none",markerfacecolor="#666",markersize=9,label="MATCH"),
                   Line2D([0],[0],marker="x",color="#666",markersize=9,label="NO MATCH"),
                   Line2D([0],[0],color="#212b36",lw=3,label="Mean, all trials")],frameon=False)
saveplot(fig,"04-trial-token-distribution","One point per trial; crosses are failed attempts whose cost is included")

In [ ]:
tmat=func.pivot(index="target",columns="arm",values="mean_tokens").reindex(columns=ARMS)/1000
fig,ax=plt.subplots(figsize=(10,4.3))
img=ax.imshow(tmat.to_numpy(),cmap="YlOrBr",aspect="auto")
for i,t in enumerate(tmat.index):
    for j,a in enumerate(ARMS):
        ax.text(j,i,f"{tmat.iloc[i,j]:,.0f}k",ha="center",va="center",fontsize=11)
ax.set_xticks(np.arange(len(ARMS)),[LABELS[a] for a in ARMS]);ax.set_yticks(np.arange(len(tmat.index)),tmat.index)
ax.set_title("Mean input + output tokens per attempt, within each function")
fig.colorbar(img,ax=ax,label="Tokens (thousands)",fraction=.035,pad=.03)
saveplot(fig,"05-function-tokens-heatmap","Mean token consumption across all three repetitions, failed attempts included")

## 4. Efficiency frontier, cache usage, and spending

The frontier plots success and **total input+output tokens divided by MATCHes**; upper-left is preferable. Success-only tokens would give an overly optimistic picture of treatments with more failures.

Cache reads/writes are displayed independently because their prices differ and the suite's published averages exclude them. OpenCode-step estimated cost and the run-budget ledger are both included, but **run-budget USD is shared-key accounting and may include other jobs**, so do not treat it as isolated trial billing.

In [ ]:
fig,ax=plt.subplots(figsize=(10,6))
for a in ARMS:
    x=agg.loc[a,"tokens_per_match_all_attempts"]/1000
    y=agg.loc[a,"success_rate"]*100
    ax.scatter(x,y,c=COLORS[a],s=200,zorder=5)
    dx,dy={"baseline":(6,1.4),"raw":(6,-2.3),"consolidated":(6,1.7),"placebo":(-65,-2.6)}[a]
    ax.annotate(f"{LABELS[a]} ({int(agg.loc[a,'matches'])}/{int(agg.loc[a,'trials'])})",
                (x,y),xytext=(dx,dy),textcoords="offset points",fontsize=10,fontweight="bold")
ax.set_xlim(max(0,agg["tokens_per_match_all_attempts"].min()/1000-32),agg["tokens_per_match_all_attempts"].max()/1000+60)
ax.set_ylim(65,99)
ax.set_xlabel("All-attempt input + output tokens per verified MATCH (thousands)")
ax.set_ylabel("Verified MATCH rate (%)")
ax.set_title("Success-efficiency frontier (upper-left is preferable)")
ax.grid(True)
saveplot(fig,"06-efficiency-frontier","All-attempt tokens per MATCH vs success rate; 9 attempts per arm")

In [ ]:
fig,ax=plt.subplots(figsize=(10,5))
xs=np.arange(len(ARMS)); w=.21
for off,col,color,label in [(-1.5,"total_input","#7aa5bd","Input"),
                            (-.5,"total_output","#d1a476","Output"),
                            (.5,"total_cache_read","#8f76b3","Cache read"),
                            (1.5,"total_cache_write","#81ae9b","Cache write")]:
    ax.bar(xs+off*w,agg[col].to_numpy()/1000,w,label=label,color=color)
ax.set_xticks(xs,[LABELS[a] for a in ARMS]);ax.set_ylabel("Cumulative token units (thousands)")
ax.set_title("Token composition across every completed trial, including failures")
ax.grid(axis="y");ax.legend(ncol=4,frameon=False)
saveplot(fig,"07-token-composition","Input, output, cache-read and cache-write counts kept separate")

costtable=agg[["db_cost_total","model_cost_total","cost_db_per_match","cost_steps_per_match"]].round(5)
display(Markdown("**Two imperfect cost sources (USD):**"));display(costtable)
if df["cost_steps_usd"].fillna(0).eq(0).all():
    print("WARNING: All OpenCode-step reported costs are zero or unavailable. Do not conclude model usage was free.")
if df["cost_db_usd"].fillna(0).eq(0).all():
    print("WARNING: Run-budget cost is zero or unavailable. Do not infer zero actual spend.")

## 5. Search effort, timings, and retrieval behavior

Checker calls are attempts to check generated code, not independent verified matches. Lower checker counts can reflect less exploration, faster convergence, or giving up earlier. Retrieval was initiated by the agent, so retrieval frequency is **not** a randomized independent explanatory variable.

In [ ]:
fig,(a1,a2)=plt.subplots(1,2,figsize=(12,4.8))
for arm in ARMS:
    r=valid[valid["arm"]==arm]
    good=r["success"].to_numpy()==1
    a1.scatter(r.loc[good,"checker_calls"],r.loc[good,"tokens"]/1000,
               color=COLORS[arm],s=58,label=LABELS[arm],alpha=.85)
    a1.scatter(r.loc[~good,"checker_calls"],r.loc[~good,"tokens"]/1000,
               marker="x",color=COLORS[arm],s=75)
    a2.scatter(r["model_steps"],r["tokens"]/1000,color=COLORS[arm],alpha=.78,s=55)
a1.set_xlabel("Agent checker calls");a2.set_xlabel("Model steps")
for ax in (a1,a2): ax.set_ylabel("Input + output tokens (thousands)");ax.grid(True)
a1.set_title("Checker usage versus tokens");a2.set_title("Model steps versus tokens")
a1.legend(frameon=False,fontsize=9)
fig.suptitle("Agent effort, all outcomes shown; × denotes NO MATCH in left panel")
saveplot(fig,"08-checker-steps","Checker calls and model steps vs total input/output tokens")

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4.8))
for i,a in enumerate(ARMS):
    r=valid[valid["arm"]==a]
    vals=r["elapsed_seconds"].dropna()/60
    if len(vals):
        axes[0].scatter(np.full(len(vals),i)+np.linspace(-.12,.12,len(vals)),vals,
                        color=COLORS[a],s=55,alpha=.85)
        axes[0].hlines(vals.median(),i-.2,i+.2,color="#222",lw=2)
    axes[1].bar(i,agg.loc[a,"total_checker_calls"],color=COLORS[a],width=.7)
axes[0].set_xticks(range(4),[LABELS[a] for a in ARMS],rotation=12)
axes[1].set_xticks(range(4),[LABELS[a] for a in ARMS],rotation=12)
axes[0].set_ylabel("Wall-clock trial duration (minutes)")
axes[1].set_ylabel("Checker calls across 9 trials")
axes[0].set_title("Time spent per trial (median bar)")
axes[1].set_title("Total checker effort")
for ax in axes: ax.grid(axis="y")
saveplot(fig,"09-runtime-checks","Wall-clock duration includes non-model overhead; counts include failed attempts")

In [ ]:
qsum = (valid.groupby("arm",observed=True).agg(
    trials=("name","count"), queries=("brain_queries","sum"),
    findings=("findings_returned","sum"), techniques=("techniques_returned","sum"),
    adherent=("treatment_adherent", lambda s:sum(x is True or x==1 for x in s))))
qsum["queries_per_trial"] = qsum["queries"]/qsum["trials"]
display(qsum)
if not query_df.empty:
    qx=query_df.groupby("arm").agg(
        query_count=("query_number","count"), chars=("characters","sum"),
        findings=("findings","sum"), techniques=("techniques","sum"))
    display(qx)
    fig,ax=plt.subplots(figsize=(10,4.6))
    for a in ARMS:
        q=query_df[query_df["arm"]==a]
        if len(q):
            ax.hist(pd.to_numeric(q["characters"],errors="coerce").dropna(),bins=np.linspace(0,3200,18),
                    histtype="step",linewidth=2,label=f"{LABELS[a]} ({len(q)} queries)",color=COLORS[a])
    ax.set_xlabel("Characters returned by memory tool per query")
    ax.set_ylabel("Number of queries")
    ax.set_title("Retrieval exposure, not simply arm assignment")
    ax.grid(axis="y");ax.legend(frameon=False)
    saveplot(fig,"10-retrieval-exposure","Output character distribution of recorded Brain queries")
else:
    print("No retrieval queries in evidence")

## 6. Run-order and provider-drift diagnostics

Trial assignment was randomized, but runs were executed sequentially. This chart lets us see whether late or early positions systematically differ in tokens, success or runtime. It cannot prove or exclude provider/model drift, and without per-request provider identifiers it cannot attribute changes to hosts.

In [ ]:
fig,axes=plt.subplots(2,1,figsize=(12,8),sharex=True)
for a in ARMS:
    sub=valid[valid["arm"]==a]
    success=sub["success"]==1
    for mask,marker in [(success,"o"),(~success,"x")]:
        if mask.any():
            axes[0].scatter(sub.loc[mask,"schedule_position"],sub.loc[mask,"tokens"]/1000,
                marker=marker,color=COLORS[a],s=65,label=LABELS[a] if marker=="o" else None)
    axes[1].scatter(sub["schedule_position"],sub["elapsed_seconds"]/60,
                    color=COLORS[a],s=55)
axes[0].set_ylabel("Input + output tokens (thousands)")
axes[1].set_ylabel("Elapsed minutes")
axes[1].set_xlabel("Randomized execution position (1 to 36)")
axes[0].set_title("Check for run-order confounding (× denotes NO MATCH)")
for ax in axes: ax.grid(True)
axes[0].legend(ncol=4,frameon=False)
saveplot(fig,"11-order-diagnostics","Execution order vs token cost and wall-clock time")

## 7. Uncertainty and robustness checks

With only **three** target functions, there is no reliable estimate of how these effects generalize to new functions. The following are *exploratory diagnostics*:

- **Leave-one-function-out:** recompute whole-experiment tokens per successful MATCH after dropping each target.
- **Function-stratified bootstrap:** resample the three repetitions within each function/arm to show trial-level sensitivity. The percentile ranges assume repetitions are exchangeable and **do not capture uncertainty across new functions**.
- **Within-function label permutation:** approximate randomization-test p-values for success-rate and mean all-attempt token differences. Because there are few functions and sequential runs, use p-values only as descriptive checks, not a definitive discovery claim. Multiple arms are compared without multiplicity correction.

In [ ]:
loo=[]
for excluded in [None]+sorted(valid["target"].unique().tolist()):
    subset=valid if excluded is None else valid[valid["target"]!=excluded]
    tab=subset.groupby("arm",observed=True).agg(wins=("success","sum"),n=("success","count"),sum_tokens=("tokens","sum")).reindex(ARMS)
    tab["token_per_match"]=tab["sum_tokens"]/tab["wins"].replace(0,np.nan)
    ref=tab.loc["baseline","token_per_match"]
    for a in ARMS[1:]:
        loo.append({"excluded_target":excluded or "None (full data)","arm":a,
                    "success":f"{int(tab.loc[a,'wins'])}/{int(tab.loc[a,'n'])}",
                    "vs_baseline_efficiency_pct":100*(tab.loc[a,"token_per_match"]/ref-1)})
loo_df=pd.DataFrame(loo)
display(loo_df.round(2))
fig,ax=plt.subplots(figsize=(10,5))
y=np.arange(len(loo_df["excluded_target"].unique()))
for j,a in enumerate(ARMS[1:]):
    part=loo_df[loo_df["arm"]==a].reset_index(drop=True)
    ax.plot(part["vs_baseline_efficiency_pct"],y+(j-1)*.17,"o-",label=LABELS[a],color=COLORS[a])
ax.axvline(0,color="#444",ls="--",lw=1)
ax.set_yticks(y,loo_df["excluded_target"].unique());ax.invert_yaxis()
ax.set_xlabel("Change in all-attempt tokens/MATCH vs baseline (%)  •  negative = fewer tokens")
ax.set_title("Leave-one-function-out: does the result depend on one function?")
ax.grid(axis="x");ax.legend(frameon=False)
saveplot(fig,"12-leave-one-function-out","Cost per MATCH difference after omitting one function")

In [ ]:
# Preserve exactly 3 repetitions per function/arm for stratified uncertainty tests.
TARGETS=sorted(valid["target"].unique())
balanced=all(len(valid[(valid["target"]==t)&(valid["arm"]==a)])==3 for t in TARGETS for a in ARMS)
BOOT_N=10000
PERM_N=12000
uncertainty=[]
perm_results=[]
if balanced and len(TARGETS)==3 and valid["tokens"].notna().all():
    tok=np.array([[valid[(valid["target"]==t)&(valid["arm"]==a)].sort_values("repeat")["tokens"].to_numpy()
                   for a in ARMS] for t in TARGETS],dtype=float)
    win=np.array([[valid[(valid["target"]==t)&(valid["arm"]==a)].sort_values("repeat")["success"].to_numpy()
                   for a in ARMS] for t in TARGETS],dtype=float)
    rng=np.random.default_rng(20261119)
    draws=rng.integers(0,3,size=(BOOT_N,len(TARGETS),len(ARMS),3))
    sampled_tok=np.take_along_axis(np.broadcast_to(tok,draws.shape),draws,axis=-1)
    sampled_win=np.take_along_axis(np.broadcast_to(win,draws.shape),draws,axis=-1)
    boot_mean_tok=sampled_tok.mean(axis=(1,3))
    boot_success=sampled_win.mean(axis=(1,3))
    boot_sums=sampled_tok.sum(axis=(1,3))
    boot_wins=sampled_win.sum(axis=(1,3))
    with np.errstate(divide="ignore",invalid="ignore"):
        boot_tpm=boot_sums/boot_wins
    for j,a in enumerate(ARMS[1:],start=1):
        for metric,data in [("success_rate_pp",100*(boot_success[:,j]-boot_success[:,0])),
                            ("all_attempt_mean_tokens",boot_mean_tok[:,j]-boot_mean_tok[:,0]),
                            ("tokens_per_match_percent",100*(boot_tpm[:,j]/boot_tpm[:,0]-1))]:
            finite=data[np.isfinite(data)]
            observed=(100*(agg.loc[a,"success_rate"]-agg.loc["baseline","success_rate"]) if metric=="success_rate_pp"
                      else agg.loc[a,"mean_tokens_all"]-agg.loc["baseline","mean_tokens_all"] if metric=="all_attempt_mean_tokens"
                      else 100*(agg.loc[a,"tokens_per_match_all_attempts"] / agg.loc["baseline","tokens_per_match_all_attempts"]-1))
            uncertainty.append({"arm":a,"metric":metric,"observed":observed,
                                "bootstrap_low":np.quantile(finite,.025) if len(finite) else np.nan,
                                "bootstrap_high":np.quantile(finite,.975) if len(finite) else np.nan,
                                "finite_draws":len(finite)})
    observed_success=win.mean(axis=(0,2)); observed_tok=tok.mean(axis=(0,2))
    null_success=np.empty((PERM_N,3));null_tok=np.empty((PERM_N,3))
    for b in range(PERM_N):
        fake_s=np.zeros((3,4));fake_t=np.zeros((3,4))
        for i in range(3):
            p=rng.permutation(12)
            vals_win=win[i].reshape(12)[p].reshape(4,3)
            vals_tok=tok[i].reshape(12)[p].reshape(4,3)
            fake_s[i]=vals_win.mean(axis=1)
            fake_t[i]=vals_tok.mean(axis=1)
        for j in range(1,4):
            null_success[b,j-1]=fake_s[:,j].mean()-fake_s[:,0].mean()
            null_tok[b,j-1]=fake_t[:,j].mean()-fake_t[:,0].mean()
    for j,a in enumerate(ARMS[1:],start=1):
        obs_s=observed_success[j]-observed_success[0]
        obs_t=observed_tok[j]-observed_tok[0]
        perm_results.append({"arm":a,"success_diff_pp":obs_s*100,
                             "two_sided_p_success":(1+np.sum(np.abs(null_success[:,j-1])>=abs(obs_s)-1e-12))/(PERM_N+1),
                             "mean_token_diff":obs_t,
                             "two_sided_p_mean_tokens":(1+np.sum(np.abs(null_tok[:,j-1])>=abs(obs_t)-1e-9))/(PERM_N+1)})
    uncertainty_df=pd.DataFrame(uncertainty)
    perm_df=pd.DataFrame(perm_results)
    display(Markdown("**Function-stratified bootstrap sensitivity intervals (not population confidence intervals):**"))
    display(uncertainty_df.round(2))
    display(Markdown("**Exploratory within-function permutation tests:**"))
    display(perm_df.round(4))
else:
    print("Stratified bootstrap/permutation unavailable: unequal/missing repetitions or missing tokens.")
    uncertainty_df=pd.DataFrame()
    perm_df=pd.DataFrame()

In [ ]:
if not uncertainty_df.empty:
    fig,axes=plt.subplots(1,2,figsize=(12,5.2))
    for k,metric in enumerate(["all_attempt_mean_tokens","success_rate_pp"]):
        s=uncertainty_df[uncertainty_df["metric"]==metric].reset_index(drop=True)
        xx=s["observed"].to_numpy()/ (1000 if k==0 else 1)
        lo=s["bootstrap_low"].to_numpy()/ (1000 if k==0 else 1)
        hi=s["bootstrap_high"].to_numpy()/ (1000 if k==0 else 1)
        yy=np.arange(3)
        axes[k].errorbar(xx,yy,xerr=[xx-lo,hi-xx],fmt="none",color="#333",capsize=5)
        axes[k].scatter(xx,yy,s=80,color=[COLORS[a] for a in s["arm"]],zorder=4)
        axes[k].axvline(0,color="#444",ls="--",lw=1)
        axes[k].set_yticks(yy,[LABELS[a] for a in s["arm"]]);axes[k].invert_yaxis()
        axes[k].set_xlabel("Difference vs baseline (thousand tokens)" if k==0 else "Success-rate difference (percentage points)")
        axes[k].grid(axis="x")
    axes[0].set_title("Mean input+output tokens, all trials")
    axes[1].set_title("Verified MATCH rate")
    fig.suptitle("Trial-level bootstrap: exploratory 95% sensitivity ranges")
    saveplot(fig,"13-bootstrap-uncertainty","Function-stratified resampling of repeats only; not reliable generalization to unseen functions")

## 8. Failure notes and evidence integrity

The next table preserves each recorded outcome and the independently recorded result note. This distinguishes failures from infrastructure errors or invalid protocols. Raw memory and placebo have multiple possible mechanisms, but **the query log does not show what model advice was actually accepted or why**; conclusions about changing reasoning strategy would require a separate behavioral coding study.

In [ ]:
display(df[["target","arm","repeat","outcome","tokens","cache_read_tokens",
           "checker_calls","model_steps","elapsed_seconds","brain_queries","result_note"]]
        .sort_values(["target","arm","repeat"]).reset_index(drop=True))
summary_failures=(df[df["valid_trial"] & (df["success"]!=1)]
                  [["target","arm","repeat","status","diff_bytes","result_note","tokens","checker_calls"]])
display(Markdown("**Non-MATCH results:**"));display(summary_failures)
if not step_df.empty:
    display(Markdown(f"**Model-step records available:** {len(step_df):,}"))
if not query_df.empty:
    display(Markdown(f"**Retrieval events available:** {len(query_df):,}"))

## 9. Export for the research report

The notebook writes **`~/sad-brain-study/sad-brain-study.zip`** with a static HTML chart dashboard, per-trial and per-step CSVs, treatment metrics, sensitivity diagnostics, source evidence JSON, and every PNG figure. Upload this ZIP to ChatGPT for a detailed research write-up. It contains retrieval queries and diagnostic notes from the exporter; **review it before sharing**. An executed `.ipynb` or Jupyter `File → Save and Export Notebook As → HTML` is also useful if you want to share cell outputs.

In [ ]:
# Machine-readable tables and reproducibility details.
df.to_csv(OUT/"trial_metrics.csv",index=False)
agg.to_csv(OUT/"arm_summary.csv")
func.to_csv(OUT/"function_summary.csv",index=False)
loo_df.to_csv(OUT/"leave_one_function_out.csv",index=False)
if not step_df.empty: step_df.to_csv(OUT/"model_steps.csv",index=False)
if not query_df.empty: query_df.to_csv(OUT/"retrieval_events.csv",index=False)
if not uncertainty_df.empty: uncertainty_df.to_csv(OUT/"uncertainty.csv",index=False)
if not perm_df.empty: perm_df.to_csv(OUT/"permutation_tests.csv",index=False)

summary_obj={
    "description":"Exploratory 36-trial known-solvable SAD Brain ablation; report writer should inspect data and caveats",
    "suite":evidence["suite"],
    "counts":{"trials":len(df),"valid":len(valid),"invalid":len(df)-len(valid),
              "functions":int(valid["target"].nunique()),"arms":ARMS},
    "arms":json.loads(agg.reset_index().to_json(orient="records")),
    "function_arm":json.loads(func.to_json(orient="records")),
    "leave_one_function_out":json.loads(loo_df.to_json(orient="records")),
    "bootstrap":json.loads(uncertainty_df.to_json(orient="records")),
    "permutation":json.loads(perm_df.to_json(orient="records")),
    "warnings":evidence.get("warnings",[]),
    "caveats":["Only 3 known-solvable functions and 3 repetitions per arm/function",
       "Function-stratified bootstrap does not assess generalization to unseen functions",
       "Cache-read/write tokens are distinct billing categories",
       "Shared-key run-budget spend may include other tasks",
       "OpenCode cost may be absent and must not be interpreted as free",
       "Placebo contains real, low-overlap findings, not meaningless text",
       "No direct causal test of changing strategy, codegen.md, or new unseen functions",
       "Target semantic equivalence is not guaranteed by masked byte fingerprint alone"]
}
(OUT/"analysis_summary.json").write_text(json.dumps(summary_obj,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")

styles="""
body{font-family:system-ui,Arial,sans-serif;max-width:1120px;margin:48px auto;padding:0 20px;color:#26323c;line-height:1.5}
h1{letter-spacing:-.03em}h2{margin-top:40px}table{border-collapse:collapse;width:100%;font-size:14px;margin:18px 0}
th,td{padding:8px 10px;text-align:right;border-bottom:1px solid #e4e8ec}th:first-child,td:first-child{text-align:left}
section{margin:38px 0}figure{margin:22px 0 34px}img{max-width:100%;height:auto}figcaption{font-size:13px;color:#56616c}
small{color:#626e79}.top{background:#f4f7f9;padding:18px 22px;border-radius:8px}.notes{font-size:14px;color:#58636b}
"""
shown=agg[["matches","trials","success_rate","tokens_per_successful_run","tokens_per_match_all_attempts"]].copy()
shown["success_rate"]=shown["success_rate"].map(lambda x:f"{x:.1%}")
for col in ("tokens_per_successful_run","tokens_per_match_all_attempts"):
    shown[col]=shown[col].map(fmtint)
shown.columns=["Matches","Trials","Success rate","Tokens per successful trial","All-attempt tokens per MATCH"]
shown.index=[LABELS[a] for a in ARMS]
intro=("<h1>SAD external-memory ablation: evidence dashboard</h1>"
       "<div class='top'><strong>Exploratory experiment, not a causal verdict.</strong> "
       "Three known-solvable functions, four arms, three repetitions each. "
       "All-attempt cost-per-MATCH includes failed trials. Cache and shared-key costs are separate.</div>")
sections=[f"<section><h2>Primary outcomes</h2>{shown.to_html(escape=True)}</section>"]
for plot in PLOTS:
    sections.append("<section><figure><img alt='"+html.escape(plot["description"],quote=True)+
                    "' src='figures/"+html.escape(plot["file"],quote=True)+"'>"+
                    "<figcaption>"+html.escape(plot["description"])+"</figcaption></figure></section>")
sections.append("<section><h2>Interpretation guardrails</h2><p class='notes'>"+
                "</p><p class='notes'>".join(html.escape(s) for s in summary_obj["caveats"])+"</p></section>")
page=("<!doctype html><html lang='en'><head><meta charset='utf-8'><meta name='viewport' content='width=device-width,initial-scale=1'>"+
     "<title>SAD Brain ablation evidence dashboard</title><style>"+styles+"</style></head><body>"+
     intro+"".join(sections)+"</body></html>")
(OUT/"dashboard.html").write_text(page,encoding="utf-8")

zpath=OUT/"sad-brain-study.zip"
with zipfile.ZipFile(zpath,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=7) as zf:
    for path in sorted(OUT.rglob("*")):
        if path.is_file() and path!=zpath:
            zf.write(path,arcname=path.relative_to(OUT))
print("Research bundle:",zpath)
print("Bundle size:",round(zpath.stat().st_size/1024/1024,2),"MiB")
print("Dashboard:",OUT/"dashboard.html")
print("Charts:",len(PLOTS),"| Trials:",len(df),"| Model steps:",len(step_df),"| Retrieval queries:",len(query_df))
print("Please review the exported notes and retrieval queries before uploading the ZIP.")